# NB-06 — Conversation Memory

**Learning goal:** Make SmartIntake remember what the user has already said across multiple turns. A user who provides `query_type` in turn 1 should not be asked for it again in turn 3.

**Concepts covered**
- `ConversationBufferMemory` — stores the full conversation
- `ConversationSummaryMemory` — summarises older turns to save tokens
- When to switch between them
- Building a multi-turn session loop
- The three required paths: happy, partial, fallback

## Cell 1 — Setup

In [ ]:
from dotenv import load_dotenv
import os
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.chains import ConversationChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    max_tokens=512
)

## Cell 2 — ConversationBufferMemory

In [ ]:
# BufferMemory keeps the full conversation history
memory = ConversationBufferMemory(return_messages=True)

# Manually simulate turns to understand what memory stores
memory.chat_memory.add_user_message(
    "PV team here. We have a SUSAR for study NCT-2244 under ICH E2A."
)
memory.chat_memory.add_ai_message(
    "Thank you. I've noted: safety_signal, ICH_E2A, clinical. "
    "Could you confirm the urgency level and your team name?"
)
memory.chat_memory.add_user_message(
    "It's critical — 15 day EMA deadline. Team is Pharmacovigilance."
)

print("Memory contents:")
for msg in memory.chat_memory.messages:
    role = "User" if msg.type == "human" else "Assistant"
    print(f"  [{role}] {msg.content}")

## Cell 3 — Multi-Turn Intake Loop (Simulated)

In [ ]:
INTAKE_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
You are collecting five fields: query_type, regulation_ref, product_area,
urgency, submitting_team.

Extract as many fields as you can from each message.
For fields you cannot extract, ask the user one question at a time.
Never ask for a field the user has already provided.
Once all five fields are collected, summarise them and ask for confirmation.

Track collected fields internally and only ask for what is still missing.
"""

def build_intake_prompt():
    return ChatPromptTemplate.from_messages([
        ("system", INTAKE_SYSTEM),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

memory = ConversationBufferMemory(memory_key="history", return_messages=True)

from langchain.chains import LLMChain

intake_chain = LLMChain(
    llm=llm,
    prompt=build_intake_prompt(),
    memory=memory,
    verbose=False
)

def chat(user_input: str):
    response = intake_chain.invoke({"input": user_input})
    print(f"User: {user_input}")
    print(f"SmartIntake: {response['text']}")
    print()

## Cell 4 — Happy Path (Single Turn)

In [ ]:
# Reset memory
memory.clear()

chat(
    "CMC Regulatory here. FDA issued a 483 on our Pune manufacturing site. "
    "We have 15 business days to respond."
)

## Cell 5 — Partial Path (Three Turns)

In [ ]:
# Reset memory for a fresh session
memory.clear()

# Turn 1: partial information
chat("Hi, we need to file a Type II variation for our cardiovascular product with the EMA.")

# Turn 2: answer one missing field
chat("The submitting team is Regulatory Affairs Europe.")

# Turn 3: answer the remaining missing field
chat("Urgency is standard — no hard deadline yet.")

## Cell 6 — Memory Switch at 10 Turns

In [ ]:
def create_memory(turn_count: int, llm):
    """Switch from BufferMemory to SummaryMemory after 10 turns."""
    if turn_count < 10:
        print(f"Turn {turn_count}: using ConversationBufferMemory")
        return ConversationBufferMemory(memory_key="history", return_messages=True)
    else:
        print(f"Turn {turn_count}: switching to ConversationSummaryMemory")
        return ConversationSummaryMemory(
            llm=llm,
            memory_key="history",
            return_messages=True
        )

# Demonstrate the switch
for turn in [1, 5, 10, 15]:
    mem = create_memory(turn, llm)

## Cell 7 — Fallback Path

In [ ]:
# Reset memory
memory.clear()

chat("Can you check the weather in Mumbai today?")

### Key Takeaway

Memory is what makes SmartIntake a *conversation* rather than a series of disconnected API calls. `BufferMemory` is simple and works for most sessions. `SummaryMemory` is the fallback for long triage conversations where token cost becomes a concern.